# 🚀 Análise Quantitativa e Backtest: BTCUSD (VT Markets)
Este notebook foi projetado para execução direta no **Google Colab** utilizando os dados históricos massivos extraídos via MetaTrader 5.

In [1]:
# 1. Instalação de dependências adicionais (caso necessário)
!pip install --quiet pyarrow fastparquet plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.8 MB/s eta 0:00:00


In [2]:
# 2. Carregamento dos dados
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Se estiver no Colab e quiser fazer upload direto:
try:
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
except Exception:
    filename = 'data/BTCUSD_M15.parquet'

print(f"Carregando arquivo: {filename}")
if filename.endswith('.parquet'):
    df = pd.read_parquet(filename)
elif filename.endswith('.csv.gz') or filename.endswith('.csv'):
    df = pd.read_csv(filename)
else:
    raise ValueError("Formato não suportado")

df['datetime'] = pd.to_datetime(df['datetime'])
df.sort_values('datetime', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Total de barras carregadas: {len(df):,}")
print(f"Período: {df['datetime'].min()} até {df['datetime'].max()}")
display(df.head())

Saving BTCUSD_M15.parquet to BTCUSD_M15.parquet
Carregando arquivo: BTCUSD_M15.parquet
Total de barras carregadas: 88,776
Período: 2024-02-20 22:45:00+00:00 até 2026-09-30 04:15:00+00:00


,datetime,open,high,low,close,tick_volume,spread,real_volume,time
0,2024-02-20 22:45:00+00:00,52056.33,52082.99,52019.00,52019.00,139,1001,0,1708469100
1,2024-02-20 23:00:00+00:00,52019.00,52153.95,52001.00,52152.87,293,1001,0,1708470000
2,2024-02-20 23:15:00+00:00,52152.87,52191.72,52104.47,52148.86,150,1001,0,1708470900
3,2024-02-20 23:30:00+00:00,52148.86,52252.71,52118.97,52252.71,181,1001,0,1708471800
4,2024-02-20 23:45:00+00:00,52252.71,52299.75,52163.65,52163.65,188,1001,0,1708472700


### 3. Engenharia de Features Quantitativas (PVSRA + Médias + ATR)

In [6]:
# Médias Móveis Exponenciais (Tendência Institucional)
df['ema_20'] = df['close'].ewm(span=20, adjust=False).mean()
df['ema_50'] = df['close'].ewm(span=50, adjust=False).mean()
df['ema_200'] = df['close'].ewm(span=200, adjust=False).mean()

# ATR (Volatilidade)
atr_period = 14
df['prev_close'] = df['close'].shift(1)
df['tr1'] = df['high'] - df['low']
df['tr2'] = (df['high'] - df['prev_close']).abs()
df['tr3'] = (df['low'] - df['prev_close']).abs()
df['tr'] = df[['tr1', 'tr2', 'tr3']].max(axis=1)
df['atr'] = df['tr'].rolling(atr_period).mean()

# Indicador PVSRA (Volume Spread Analysis)
lookback = 10
df['avg_vol'] = df['tick_volume'].rolling(lookback).mean()
df['candle_spread'] = (df['close'] - df['open']).abs()
df['avg_spread'] = df['candle_spread'].rolling(lookback).mean()

# Classificação PVSRA
cond_climax = (df['tick_volume'] >= 2.0 * df['avg_vol']) | ((df['tick_volume'] * df['candle_spread']) >= (2.0 * df['avg_vol'] * df['avg_spread']))
cond_high = df['tick_volume'] >= 1.5 * df['avg_vol']

df['pvsra_type'] = 'NORMAL'
df.loc[cond_high, 'pvsra_type'] = 'HIGH'
df.loc[cond_climax, 'pvsra_type'] = 'CLIMAX'

print("Distribuição de Volume PVSRA:")
print(df['pvsra_type'].value_counts())

Distribuição de Volume PVSRA:
pvsra_type
NORMAL    74064
CLIMAX    12945
HIGH       1767
Name: count, dtype: int64


### 4. Backtest de Estratégia Conceitual (Trend Pullback + Climax Volume Absorption)

In [7]:
# Sinal simples: Compra quando tendência for de alta (EMA 50 > EMA 200) e ocorrer absorção de volume (CLIMAX/HIGH)
df['signal'] = 0
# Compra
buy_cond = (df['close'] > df['ema_200']) & (df['pvsra_type'].isin(['CLIMAX', 'HIGH'])) & (df['close'] > df['open'])
# Venda
sell_cond = (df['close'] < df['ema_200']) & (df['pvsra_type'].isin(['CLIMAX', 'HIGH'])) & (df['close'] < df['open'])

df.loc[buy_cond, 'signal'] = 1
df.loc[sell_cond, 'signal'] = -1

# Retorno de 1 período à frente
df['future_return'] = df['close'].pct_change().shift(-1)
df['strategy_return'] = df['signal'] * df['future_return']

# Curva de Capital (Cumulative Return)
df['cum_market'] = (1 + df['future_return']).cumprod()
df['cum_strategy'] = (1 + df['strategy_return']).cumprod()

total_return_pct = (df['cum_strategy'].iloc[-2] - 1) * 100
win_rate = (df.loc[df['signal'] != 0, 'strategy_return'] > 0).mean() * 100
trades = (df['signal'] != 0).sum()

print(f"--- RESULTADOS PRELIMINARES ---")
print(f"Trades acionados: {trades:,}")
print(f"Taxa de acerto (1-bar horizon): {win_rate:.2f}%")
print(f"Retorno acumulado: {total_return_pct:.2f}%")

--- RESULTADOS PRELIMINARES ---
Trades acionados: 9,047
Taxa de acerto (1-bar horizon): 45.48%
Retorno acumulado: 46.87%


### 5. Visualização Interativa do Comportamento de Preço e Volume

In [8]:
# Visualizar as últimas 300 barras de forma interativa
subset = df.tail(300).copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03, row_heights=[0.7, 0.3])

# Candlestick
fig.add_trace(go.Candlestick(x=subset['datetime'], open=subset['open'], high=subset['high'], low=subset['low'], close=subset['close'], name='BTCUSD'), row=1, col=1)

# EMAs
fig.add_trace(go.Scatter(x=subset['datetime'], y=subset['ema_20'], line=dict(color='orange', width=1), name='EMA 20'), row=1, col=1)
fig.add_trace(go.Scatter(x=subset['datetime'], y=subset['ema_50'], line=dict(color='cyan', width=1), name='EMA 50'), row=1, col=1)
fig.add_trace(go.Scatter(x=subset['datetime'], y=subset['ema_200'], line=dict(color='purple', width=1.5), name='EMA 200'), row=1, col=1)

# Volume colorido por PVSRA
colors = {'NORMAL': '#888888', 'HIGH': '#2196F3', 'CLIMAX': '#E91E63'}
vol_colors = [colors[t] for t in subset['pvsra_type']]
fig.add_trace(go.Bar(x=subset['datetime'], y=subset['tick_volume'], marker_color=vol_colors, name='Volume PVSRA'), row=2, col=1)

fig.update_layout(title='BTCUSD - Análise Institucional PVSRA e Médias', xaxis_rangeslider_visible=False, height=600)
fig.show()